## ETL dei dati
### Comincio dal dataset NYT_news

In [1]:
# Carico i dati
import pandas as pd

df_nyt = pd.read_csv("NYT_news.csv", sep=",", encoding="utf-8")

In [2]:
# Informazioni generali sul dataframe
# Nota sui nomi colonna: il file non ha 'autore'/'data'/'sezione'/'headline'/'body text'
# ma writer_name, pub_date, section_name, main_headline, body_text (e _key come url dell'articolo)
print(df_nyt.dtypes)
print(df_nyt.shape[0])

_key                 object
writer_name          object
pub_date             object
snippet              object
word_count            int64
score               float64
source               object
section_name         object
type_of_material     object
document_type        object
main_headline        object
print_headline       object
body_text            object
dtype: object
66038


In [3]:
# calcolo i valori mancanti per ogni colonna
missing_count_nyt = {}
for col in df_nyt.columns:
    missing_count = df_nyt[col].isnull().sum()
    missing_count_nyt[col] = missing_count
    print(f"Colonna: {col}, Valori mancanti: {missing_count}")

Colonna: _key, Valori mancanti: 0
Colonna: writer_name, Valori mancanti: 14684
Colonna: pub_date, Valori mancanti: 0
Colonna: snippet, Valori mancanti: 28881
Colonna: word_count, Valori mancanti: 0
Colonna: score, Valori mancanti: 66038
Colonna: source, Valori mancanti: 0
Colonna: section_name, Valori mancanti: 0
Colonna: type_of_material, Valori mancanti: 865
Colonna: document_type, Valori mancanti: 0
Colonna: main_headline, Valori mancanti: 4
Colonna: print_headline, Valori mancanti: 2317
Colonna: body_text, Valori mancanti: 1824


In [4]:
# Analizzo le righe duplicate per _key (url) e body_text insieme
duplicate_count_nyt = df_nyt.duplicated(subset=["_key", "body_text"]).sum()
print(f"Righe duplicate (stesso url e stesso body_text): {duplicate_count_nyt}")

# rimuovo le righe duplicate solo se _key e body_text sono uguali, mantenendo la prima occorrenza
df_nyt = df_nyt.drop_duplicates(subset=["_key", "body_text"], keep="first")

Righe duplicate (stesso url e stesso body_text): 509


In [5]:
# File di ispezione: solo le righe in cui mancano SIA sezione SIA type of material
# (calcolato prima di eliminare le righe con valori mancanti, per poterle ispezionare)
df_nyt_nosez = df_nyt[df_nyt["section_name"].isnull() & df_nyt["type_of_material"].isnull()]
df_nyt_nosez.to_csv("NYT_clean_nosez.csv", index=False, encoding="utf-8")
print(f"Righe senza sezione e senza type of material: {len(df_nyt_nosez)}")

Righe senza sezione e senza type of material: 0


In [6]:
# Rimuovo le righe solo se mancano data, type of material, sezione, headline o body text
# (l'autore puo' mancare, non lo considero per questo file)
colonne_obbligatorie = ["pub_date", "type_of_material", "section_name", "main_headline", "body_text"]
df_nyt_clean = df_nyt.dropna(subset=colonne_obbligatorie)

# Tengo solo le colonne utili per l'analisi
df_nyt_clean = df_nyt_clean[["writer_name", "pub_date", "type_of_material", "section_name", "main_headline", "body_text"]]

# Salvo il dataframe pulito in un nuovo file CSV
df_nyt_clean.to_csv("NYT_news_cleaned.csv", index=False, encoding="utf-8")
print(df_nyt_clean.shape)

(62856, 6)


In [7]:
# Variante che rimuove le righe anche se manca l'autore
df_nyt_clean_autors = df_nyt_clean.dropna(subset=["writer_name"])
df_nyt_clean_autors.to_csv("NYT_clean_autors.csv", index=False, encoding="utf-8")
print(df_nyt_clean_autors.shape)

(49390, 6)


In [ ]:
# vedo i valori unici della colonna section_name
tema_nyt = df_nyt_clean['section_name'].unique()
print(tema_nyt)


In [ ]:
# raccolgo tutte le variabili calcolate durante l'analisi in un unico dizionario,
# cosi' da poterle riutilizzare facilmente in un altro notebook
extra_nyt = {
    "missing_count": missing_count_nyt,
    "duplicate_count": duplicate_count_nyt,
    "temi": tema_nyt,
}

#salvo il dizionario in un file json all'interno della cartella del progetto, cosi' da poterlo riutilizzare in un altro notebook


In [ ]:
import json
import numpy as np


def _json_default(obj):
    """Converte in tipi standard i valori non nativamente serializzabili in JSON
    (Timestamp di pandas, DataFrame, array/interi/float numpy)."""
    if isinstance(obj, pd.DataFrame):
        return obj.to_dict(orient="records")
    if isinstance(obj, pd.Series):
        return obj.to_dict()
    if isinstance(obj, pd.Timestamp):
        return obj.isoformat()
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        return float(obj)
    raise TypeError(f"Tipo non serializzabile in JSON: {type(obj)}")


with open("extra_nyt.json", "w", encoding="utf-8") as f:
    json.dump(extra_nyt, f, ensure_ascii=False, indent=2, default=_json_default)

print("Salvato extra_nyt.json")